In [1]:
import torch
import math
import torch.nn as nn
import torch.nn.functional as F

In [2]:
def scaled_dot_product_sttantion(Q, K, V, mask=None):
    d_model = Q.size(-1)
    scores= Q @ K.transpose(-1, -2) / math.sqrt(d_model)

    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))

    attention_weights = F.softmax(scores, dim=-1)
    output = attention_weights @ V

    return output, attention_weights


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, h):
        super(MultiHeadAttention, self).__init__()

        assert d_model % h == 0

        self.d_model = d_model
        self.h = h
        h_dim = d_model / h
        self.h_dim = h_dim

        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forwaerd(self, q, k, v, mask):
        batch_size = q.shape[0]
        seq_len_q = q.shape[1]
        seq_len_k = k.shape[1]

        Q = self.w_q(q).reshape(batch_size, seq_len_q, self.h, self.h_dim).transpose(1, 2)
        K = self.w_k(k).reshape(batch_size, seq_len_k, self.h, self.h_dim).transpose(1, 2)
        V = self.w_v(v).reshape(batch_size, seq_len_k, self.h, self.h_dim).transpose(1, 2)

        scaled_attention, _ = scaled_dot_product_sttantion(Q, K, V, mask)
        concat_out = scaled_attention.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        out = self.fc_out(concat_out)

        return out
        

In [3]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionwiseFeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.w_2(F.relu(self.w_1(x)))

In [4]:
class ResidualConnection(nn.Module):
    def __init__(self, dropout=0.1):
        super(ResidualConnection, self).__init__()
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(x))

class LayerNorm(nn.Module):
    def __init__(self, feature_size, epsilon=1e-9):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(feature_size))
        self.bata = nn.Parameter(torch.zeros(feature_size))
        self.epsilon = epsilon

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.epsilon) + self.bata

class SublayerConnection(nn.Module):
    def __init__(self, feature_size, dropout=0.1, epsilon=1e-9):
        super(SublayerConnection, self).__init__()
        self.residual = ResidualConnection(dropout)
        self.norm = LayerNorm(feature_size, epsilon)

    def forward(self, x, sublayer):
        return self.norm(self.residual(x, sublayer))

In [5]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff, dropout):
        """
        d_model:嵌入维度
        h:头数
        d_ff:前馈层的隐藏层维度
        """
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, h)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)
        self.sublayer = nn.ModuleList([SublayerConnection(d_model, dropout) for _ in range(2)])
        self.d_model = d_model

    def forward(self, x, src_mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, src_mask))
        x = self.sublayer[1](x, self.feed_forward)
        return x

## 以上为编码器层

In [6]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, h, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, h)
        self.sublayer = nn.ModuleList([SublayerConnection(d_model, dropout) for _ in range(3)])
        self.cross_attn = MultiHeadAttention(d_model, h)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)
        self.d_model = d_model

    def forward(self, x, memory, src_mask, tgt_mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.cross_attn(x, memory, memory, src_mask))
        x = self.sublayer[2](x, self.feed_forward)
        return x

## 解码器层

In [7]:
class Encoder(nn.Module):
    def __init__(self, d_model, N, h, d_ff, dropout=0.1):
        super(Encoder, self).__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, h, d_ff, dropout) for _ in range(N)
        ])
        self.norm = LayerNorm(d_model)

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, d_model, N, h, d_ff, dropout=0.1):
        super(Decoder, self).__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, h, d_ff, dropout) for _ in range(N)
        ])
        self.norm = LayerNorm(d_model)
    
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


## 编码器与解码器

In [8]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super(Embedding, self).__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.scale_factor = math.sqrt(d_model)

    def forward(self, x):
        return self.embedding(x) * self.scale_factor

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)
    
class SourceEmbedding(nn.Module):
    def __init__(self, src_vocab_size, d_model, dropout=0.1):
        super(SourceEmbedding, self).__init__()
        self.embed = Embedding(src_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout)

    def forward(self, x):
        x = self.embed(x)
        return self.positional_encoding(x)

class TargetEmbedding(nn.Module):
    def __init__(self, tgt_vocab_size, d_model, dropout=0.1):
        super(TargetEmbedding, self).__init__()
        self.embed = Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout)

    def forward(self, x):
        x = self.embed(x)
        return self.positional_encoding(x)


In [9]:
def create_padding_mask(seq, pad_token_id=0):
    mask = (seq != pad_token_id).unsqueeze(1).unsequeeze(2)
    return mask

def create_ahead_mask(size):
    mask = torch.tril(torch.ones(size, size)).type(torch.bool)
    return mask

def create_decoder_mask(tgt_seq, pad_token_id=0):
    padding_mask = create_padding_mask(tgt_seq, pad_token_id)
    look_ahead_mask = create_ahead_mask(tgt_seq.size(1)).to(tgt_seq.device)

    combined_mask = look_ahead_mask.unsqueeze(0) & padding_mask
    return combined_mask

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, N, h, d_ff, dropout=0.1):
        super(Transformer, self).__init__()

        self.src_embedding = SourceEmbedding(src_vocab_size, d_model, dropout)
        self.tgt_embedding = TargetEmbedding(tgt_vocab_size, d_model, dropout)

        self.encoder = Encoder(d_model, N, h, d_ff, dropout)
        self.decoder = Decoder(d_model, N, h, d_ff, dropout)

        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        src_mask = create_padding_mask(src)
        tgt_mask = create_decoder_mask(tgt)

        enc_output = self.encoder(self.src_embedding(src), src_mask)
        
        dec_output = self.decoder(self.tgt_embedding(tgt), enc_output, src_mask, tgt_mask)

        output = self.fc_out(dec_output)

        return output

## 完整实现

In [10]:
src_vocab_size = 5000
tgt_vocab_size = 5000

d_model = 512
N = 6
h = 8
d_ff = 2048
dropout = 0.1

model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    N=N,
    h=h,
    d_ff=d_ff,
    dropout=dropout
)

print(model)

Transformer(
  (src_embedding): SourceEmbedding(
    (embed): Embedding(
      (embedding): Embedding(5000, 512)
    )
    (positional_encoding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (tgt_embedding): TargetEmbedding(
    (embed): Embedding(
      (embedding): Embedding(5000, 512)
    )
    (positional_encoding): PositionalEncoding(
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (self_attn): MultiHeadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (fc_out): Linear(in_features=512, out_features=512, bias=True)
        )
        (feed_forward): PositionwiseFeedForward(
          (w_1): Linear(in_features=512, out_features=2048, bias=True)
          (w_2): Linear(in_feature